In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [1]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 
import numpy as np

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [20]:
# Alice and Bob each choose an operator randomly from their respective sets of operators.
# Alice's set is {A0, A1, A2} and Bob's set is {B0, B1, B2}.

ALICE_BASIS = {
    0: "X",  # A1
    1: "W",  # A2
    2: "Z",  # A3
}

BOB_BASIS = {
    0: "W",  # B1
    1: "Z",  # B2
    2: "V",  # B3
}

ATTACKER_BASIS = {
    0: "Z",
    1: "W",
    2: "V"
}

# If Alice chooses A3 and Bob chooses B2 then they are both measuring in
# the standard basis and their results are guaranteed to be opposite.
# Similarly if Alice chooses A2 and Bob chooses B1 then they are both
# measuring in the W basis and their results are also guaranteed to be oppo
# site.

# If Alice’s and Bob’s choices correspond to terms in the definition of S
# above, then they use the results as part of the calculation of S. This covers
# the choices (A1,B1), (A1,B3), (A3,B1) and (A3,B3). There are three other
# possible combinations of measurement: (A1,B2), (A2,B2) and (A2,B3). The
# results of these measurements are discarded

S_CALCULATION_CHOICES = (
    (0, 0),  # A1, B1
    (0, 2),  # A1, B3
    (2, 0),  # A3, B1
    (2, 2),  # A3, B3
)
S_CALCULATION_SET = set(S_CALCULATION_CHOICES)

DISCARDED_CHOICES = {
    (0, 1),  # A1, B2
    (1, 1),  # A2, B2
    (1, 2),  # A2, B3
}

KEY_CHOICES = {
    (1, 0),  # A2, B1
    (2, 1),  # A3, B2
}

SIMULATOR = BasicSimulator()

basisChoice = QuantumCircuit(1, 1)
basisU = np.array(
    [[1 / np.sqrt(3), -np.sqrt(2 / 3)], [np.sqrt(2 / 3), 1 / np.sqrt(3)]]
)
basisChoice.unitary(Operator(basisU), [0])
basisChoice.measure(0, 0)
COMPILED_BASIS_CHOICE = transpile(basisChoice, SIMULATOR)

basisChoice2 = QuantumCircuit(1, 1)
basisChoice2.h(0)
basisChoice2.measure(0, 0)
COMPILED_BASIS_CHOICE2 = transpile(basisChoice2, SIMULATOR)


def bitConvert(bit):
    return 1 if bit == 0 else -1


def simulate(compiled_circuit):
    result = SIMULATOR.run(compiled_circuit, shots=1).result()
    return next(iter(result.get_counts()))


def basisGenerator():
    if int(simulate(COMPILED_BASIS_CHOICE)[0]) == 1:
        return 1 if int(simulate(COMPILED_BASIS_CHOICE2)[0]) == 1 else 2
    return 0


def generateEntangledPair():
    qc = QuantumCircuit(2, 3)
    qc.h(0)
    qc.cx(0, 1)
    qc.z(1)
    qc.x(0)
    return qc


def attackerMeasurement(qc, basis):
    if basis == "Z":
        qc.measure(1, 2)
    elif basis == "W":
        qc.ry(3 * np.pi / 4, 1)
        qc.measure(1, 2)
    elif basis == "V":
        qc.ry(-3 * np.pi / 4, 1)
        qc.measure(1, 2)
    else:
        raise ValueError("Invalid basis")


def aliceMeasurement(qc, basis):
    if basis == "X":
        qc.h(0)
    elif basis == "W":
        qc.ry(3 * np.pi / 4, 0)
    elif basis == "Z":
        pass
    else:
        raise ValueError("Invalid basis")


def bobMeasurement(qc, basis):
    if basis == "W":
        qc.ry(3 * np.pi / 4, 1)
    elif basis == "Z":
        pass
    elif basis == "V":
        qc.ry(-3 * np.pi / 4, 1)
    else:
        raise ValueError("Invalid basis")


def measureWithAttacker():
    compiled = {}
    for attackerChoice, attackerBasis in ATTACKER_BASIS.items():
        for aliceChoice, aliceBasis in ALICE_BASIS.items():
            for bobChoice, bobBasis in BOB_BASIS.items():
                qc = generateEntangledPair()
                attackerMeasurement(qc, attackerBasis)
                aliceMeasurement(qc, aliceBasis)
                bobMeasurement(qc, bobBasis)

                qc.measure(0, 0) # Alice's measurement
                qc.measure(1, 1) # Bob's measurement

                compiled[(attackerChoice, aliceChoice, bobChoice)] = transpile(qc, SIMULATOR)
    return compiled


COMPILED_ATTACKED_MEASUREMENT_CIRCUITS = measureWithAttacker()


def calculateS(sSum, sCount):
    # S = | <XW> - <XV> + <ZW> + <ZV> |
    coefficients = {
        (0, 0): 1,
        (0, 2): -1,
        (2, 0): 1,
        (2, 2): 1,
    }

    terms = {}
    for pair in S_CALCULATION_CHOICES:
        terms[pair] = sSum[pair] / sCount[pair] if sCount[pair] > 0 else 0

    print("XW =", terms[(0, 0)])
    print("XV =", terms[(0, 2)])
    print("ZW =", terms[(2, 0)])
    print("ZV =", terms[(2, 2)])

    return abs(sum(coefficients[pair] * terms[pair] for pair in S_CALCULATION_CHOICES))


def E91_WithAttacker(N):
    steps = int(np.ceil(9 * N / 2))

    results = {
        "attacker_basis": [],
        "alice_basis": [],
        "bob_basis": [],
        "attacker_measurements": [],
        "alice_measurements": [],
        "bob_measurements": [],
    }

    sSum = {pair: 0 for pair in S_CALCULATION_CHOICES}
    sCount = {pair: 0 for pair in S_CALCULATION_CHOICES}

    for _ in range(steps):
        aliceChoice = basisGenerator()
        bobChoice = basisGenerator()
        attackerChoice = basisGenerator()  
        choicePair = (aliceChoice, bobChoice)

        results["attacker_basis"].append(ATTACKER_BASIS[attackerChoice])
        results["alice_basis"].append(ALICE_BASIS[aliceChoice])
        results["bob_basis"].append(BOB_BASIS[bobChoice])

        bitstring = simulate(COMPILED_ATTACKED_MEASUREMENT_CIRCUITS[(attackerChoice, aliceChoice, bobChoice)])
        aliceBit = int(bitstring[2])
        bobBit = int(bitstring[1])
        attackerBit = int(bitstring[0])
        results["attacker_measurements"].append(attackerBit)

        if choicePair in KEY_CHOICES:
            results["alice_measurements"].append(aliceBit)
            results["bob_measurements"].append(bobBit)

        elif choicePair in S_CALCULATION_SET:
            sSum[choicePair] += bitConvert(aliceBit) * bitConvert(bobBit)
            sCount[choicePair] += 1
        elif choicePair in DISCARDED_CHOICES:
            continue

    s = calculateS(sSum, sCount)
    print("sCount =", sCount)
    return results, s

In [21]:
if __name__ == "__main__":
    N = 1000
    results, s = E91_WithAttacker(N)

    print(f"Rounds simulated: {int(np.ceil(9 * N / 2))}")
    print(f"Key rounds kept: {len(results['alice_measurements'])}")
    print("First 20 Alice basis choices:", results["alice_basis"][:20])
    print("First 20 Bob basis choices:", results["bob_basis"][:20])
    print("First 20 Attacker basis choices:", results["attacker_basis"][:20])
    print("First 20 Alice key bits:        ", results["alice_measurements"][:20])
    print("First 20 Bob key bits:          ", results["bob_measurements"][:20])
    print("First 20 Attacker key bits:     ", results["attacker_measurements"][:20])

    bobInverted = [1 - b for b in results["bob_measurements"]]
    print("Keys match (after Bob's key is inverted):", results["alice_measurements"] == bobInverted)

    correct_basis = sum(1 for a, b in zip(results["attacker_basis"], results["bob_basis"]) if a == b)
    print("Attacker guessed Bob's basis:", correct_basis, "times out of", len(results["bob_basis"]))
    print("Attacker basis guess rate:", correct_basis / len(results["bob_basis"]))

    print("S Result:", s)

XW = -0.01937984496124031
XV = 0.020491803278688523
ZW = -0.125
ZV = -0.0870488322717622
sCount = {(0, 0): 516, (0, 2): 488, (2, 0): 512, (2, 2): 471}
Rounds simulated: 4500
Key rounds kept: 1011
First 20 Alice basis choices: ['W', 'Z', 'Z', 'X', 'W', 'W', 'X', 'Z', 'Z', 'X', 'X', 'X', 'W', 'Z', 'W', 'W', 'Z', 'W', 'W', 'X']
First 20 Bob basis choices: ['Z', 'W', 'V', 'W', 'V', 'V', 'V', 'W', 'W', 'W', 'Z', 'W', 'W', 'Z', 'Z', 'W', 'Z', 'W', 'V', 'Z']
First 20 Attacker basis choices: ['V', 'Z', 'V', 'V', 'W', 'V', 'W', 'V', 'W', 'V', 'W', 'Z', 'V', 'Z', 'V', 'W', 'V', 'Z', 'Z', 'V']
First 20 Alice key bits:         [0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]
First 20 Bob key bits:           [1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1]
First 20 Attacker key bits:      [0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0]
Keys match (after Bob's key is inverted): False
Attacker guessed Bob's basis: 1511 times out of 4500
Attacker basis guess rate: 